# 02_kde_evaluation

Este notebook compara **apenas resultados do exp_015**:

- `m_30_baseline_vs_kdehist_global.csv`
- `m_30_baseline_vs_kdehist_perclass.csv`

Cada CSV já contém os dois modelos (baseline e KDE+hist), então não usa mais nada do exp_013.


In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

REPO_ROOT = Path.cwd()

GLOBAL_PATHS = [
    REPO_ROOT / 'm_30_baseline_vs_kdehist_global.csv',
    REPO_ROOT / 'experiments' / 'exp_015' / 'm_30_baseline_vs_kdehist_global.csv',
]
PERCLASS_PATHS = [
    REPO_ROOT / 'm_30_baseline_vs_kdehist_perclass.csv',
    REPO_ROOT / 'experiments' / 'exp_015' / 'm_30_baseline_vs_kdehist_perclass.csv',
]

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

global_path = first_existing(GLOBAL_PATHS)
perclass_path = first_existing(PERCLASS_PATHS)

print('global_path  :', global_path)
print('perclass_path:', perclass_path)

if global_path is None or perclass_path is None:
    raise FileNotFoundError('Não encontrei os CSVs do exp_015 em caminhos esperados.')


global_path  : /var/new_homes/julio/mestrado/mestrado-dyssyn/experiments/exp_015/m_30_baseline_vs_kdehist_global.csv
perclass_path: /var/new_homes/julio/mestrado/mestrado-dyssyn/experiments/exp_015/m_30_baseline_vs_kdehist_perclass.csv


In [2]:
global_df = pd.read_csv(global_path)
perclass_df = pd.read_csv(perclass_path)

required = {'dataset', 'modelo', 'erro'}
for name, df in [('global', global_df), ('perclass', perclass_df)]:
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{name}: faltam colunas obrigatórias {missing}')

print('Global rows   :', len(global_df), '| datasets:', global_df['dataset'].nunique())
print('PerClass rows :', len(perclass_df), '| datasets:', perclass_df['dataset'].nunique())

display(global_df.head())
display(perclass_df.head())


Global rows   : 14400 | datasets: 24
PerClass rows : 14400 | datasets: 24


,dataset,modelo,n_classes_original,erro
0,abalone.csv,MoSS_Baseline,11,0.076687
1,abalone.csv,MoSS_KDEHist,11,0.075214
2,abalone.csv,MoSS_Baseline,11,0.071555
3,abalone.csv,MoSS_KDEHist,11,0.095030
4,abalone.csv,MoSS_Baseline,11,0.077335


,dataset,modelo,n_classes_original,erro
0,abalone.csv,MoSS_Baseline_PerClass,11,0.071409
1,abalone.csv,MoSS_KDEHist_PerClass,11,0.071082
2,abalone.csv,MoSS_Baseline_PerClass,11,0.074582
3,abalone.csv,MoSS_KDEHist_PerClass,11,0.069340
4,abalone.csv,MoSS_Baseline_PerClass,11,0.074276


## Global (exp_015)


In [3]:
global_agg = (
    global_df.groupby(['dataset', 'modelo'], as_index=False)
    .agg(erro_mean=('erro', 'mean'), erro_median=('erro', 'median'), n=('erro', 'size'))
)

display(global_agg.head())


,dataset,modelo,erro_mean,erro_median,n
0,abalone.csv,MoSS_Baseline,0.065860,0.067749,300
1,abalone.csv,MoSS_KDEHist,0.069001,0.070313,300
2,academic-success.csv,MoSS_Baseline,0.131742,0.122243,300
3,academic-success.csv,MoSS_KDEHist,0.176462,0.168828,300
4,chess.csv,MoSS_Baseline,0.046167,0.048461,300


In [4]:
fig = px.box(
    global_df,
    x='modelo',
    y='erro',
    points='outliers',
    title='Global (exp_015): distribuição de erro por modelo'
)
fig.update_layout(xaxis_title='Modelo', yaxis_title='Erro absoluto')
fig.show()


In [6]:
fig = px.bar(
    global_agg,
    x='dataset',
    y='erro_mean',
    color='modelo',
    barmode='group',
    title='Global (exp_015): erro médio por dataset'
)
fig.update_xaxes(tickangle=45)
fig.update_layout(yaxis_title='Erro médio')
fig.show()


## PerClass (exp_015)


In [7]:
perclass_agg = (
    perclass_df.groupby(['dataset', 'modelo'], as_index=False)
    .agg(erro_mean=('erro', 'mean'), erro_median=('erro', 'median'), n=('erro', 'size'))
)

display(perclass_agg.head())


,dataset,modelo,erro_mean,erro_median,n
0,abalone.csv,MoSS_Baseline_PerClass,0.065088,0.065997,300
1,abalone.csv,MoSS_KDEHist_PerClass,0.059401,0.057337,300
2,academic-success.csv,MoSS_Baseline_PerClass,0.176803,0.161354,300
3,academic-success.csv,MoSS_KDEHist_PerClass,0.137020,0.120031,300
4,chess.csv,MoSS_Baseline_PerClass,0.050412,0.051360,300


In [8]:
fig = px.box(
    perclass_df,
    x='modelo',
    y='erro',
    points='outliers',
    title='PerClass (exp_015): distribuição de erro por modelo'
)
fig.update_layout(xaxis_title='Modelo', yaxis_title='Erro absoluto')
fig.show()


In [10]:
fig = px.bar(
    perclass_agg,
    x='dataset',
    y='erro_mean',
    color='modelo',
    barmode='group',
    title='PerClass (exp_015): erro médio por dataset'
)
fig.update_xaxes(tickangle=45)
fig.update_layout(yaxis_title='Erro médio')
fig.show()


## Ranking resumido (baseline vs KDE+hist)


In [11]:
def rank_by_model(agg_df, title):
    print("\n" + title)
    for model in sorted(agg_df['modelo'].unique()):
        top = (
            agg_df[agg_df['modelo'] == model]
            .sort_values('erro_mean')
            .head(15)
            [['dataset', 'modelo', 'erro_mean', 'erro_median', 'n']]
        )
        print(f"\nModelo: {model}")
        display(top)

rank_by_model(global_agg, 'Global (exp_015)')
rank_by_model(perclass_agg, 'PerClass (exp_015)')



Global (exp_015)

Modelo: MoSS_Baseline


,dataset,modelo,erro_mean,erro_median,n
20,isolet.csv,MoSS_Baseline,0.031839,0.032937,300
22,letter.csv,MoSS_Baseline,0.032051,0.034351,300
4,chess.csv,MoSS_Baseline,0.046167,0.048461,300
14,hand_digits.csv,MoSS_Baseline,0.057693,0.056170,300
10,digits.csv,MoSS_Baseline,0.057894,0.056585,300
18,image_seg.csv,MoSS_Baseline,0.065742,0.066787,300
0,abalone.csv,MoSS_Baseline,0.065860,0.067749,300
12,dry-bean.csv,MoSS_Baseline,0.066483,0.063939,300
30,obesity.csv,MoSS_Baseline,0.067173,0.068912,300
26,molecular.csv,MoSS_Baseline,0.072854,0.063513,300



Modelo: MoSS_KDEHist


,dataset,modelo,erro_mean,erro_median,n
21,isolet.csv,MoSS_KDEHist,0.033638,0.035101,300
23,letter.csv,MoSS_KDEHist,0.033785,0.035372,300
5,chess.csv,MoSS_KDEHist,0.046442,0.047970,300
1,abalone.csv,MoSS_KDEHist,0.069001,0.070313,300
11,digits.csv,MoSS_KDEHist,0.071931,0.071014,300
31,obesity.csv,MoSS_KDEHist,0.074322,0.075640,300
15,hand_digits.csv,MoSS_KDEHist,0.075045,0.071188,300
37,poker_hand.csv,MoSS_KDEHist,0.091066,0.087984,300
19,image_seg.csv,MoSS_KDEHist,0.091193,0.088198,300
13,dry-bean.csv,MoSS_KDEHist,0.092179,0.088673,300



PerClass (exp_015)

Modelo: MoSS_Baseline_PerClass


,dataset,modelo,erro_mean,erro_median,n
22,letter.csv,MoSS_Baseline_PerClass,0.035255,0.035402,300
20,isolet.csv,MoSS_Baseline_PerClass,0.035388,0.035468,300
4,chess.csv,MoSS_Baseline_PerClass,0.050412,0.051360,300
10,digits.csv,MoSS_Baseline_PerClass,0.063455,0.063531,300
14,hand_digits.csv,MoSS_Baseline_PerClass,0.064091,0.066261,300
0,abalone.csv,MoSS_Baseline_PerClass,0.065088,0.065997,300
36,poker_hand.csv,MoSS_Baseline_PerClass,0.072498,0.068362,300
38,satellite.csv,MoSS_Baseline_PerClass,0.092006,0.091766,300
18,image_seg.csv,MoSS_Baseline_PerClass,0.100997,0.089418,300
30,obesity.csv,MoSS_Baseline_PerClass,0.103151,0.093580,300



Modelo: MoSS_KDEHist_PerClass


,dataset,modelo,erro_mean,erro_median,n
21,isolet.csv,MoSS_KDEHist_PerClass,0.034372,0.035600,300
23,letter.csv,MoSS_KDEHist_PerClass,0.034736,0.035527,300
5,chess.csv,MoSS_KDEHist_PerClass,0.049184,0.048900,300
1,abalone.csv,MoSS_KDEHist_PerClass,0.059401,0.057337,300
11,digits.csv,MoSS_KDEHist_PerClass,0.073274,0.070976,300
15,hand_digits.csv,MoSS_KDEHist_PerClass,0.084060,0.086111,300
37,poker_hand.csv,MoSS_KDEHist_PerClass,0.098162,0.100417,300
45,wine-quality.csv,MoSS_KDEHist_PerClass,0.111204,0.121946,300
7,cmc.csv,MoSS_KDEHist_PerClass,0.123365,0.114744,300
39,satellite.csv,MoSS_KDEHist_PerClass,0.131611,0.131312,300


In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

# ============================
# CARREGAR TODOS OS CSVs
# ============================

results_path = Path("../exp_015")

csv_files = list(results_path.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("Nenhum CSV encontrado na pasta ../results")

dfs = []
for csv in csv_files:
    df_tmp = pd.read_csv(csv)
    df_tmp["source_file"] = csv.name  # opcional: rastrear origem
    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)

# 🔧 garantir que erro é numérico
df["erro"] = pd.to_numeric(df["erro"], errors="coerce")

# ============================
# ORDENAR MODELOS
# ============================

order = (
    df.groupby("modelo")["erro"]
    .median()
    .sort_values()
    .index
    .tolist()
)

# ============================
# MAPA DE CORES
# ============================

color_map = {
    "baseline_ultra":  "#EF553B",
    "shape_ultra":  "#EF553B",
    "tail_ultra":  "#EF553B",
    "divergence_ultra": "#EF553B",
    "qderiv_ultra": "#EF553B",
    "baseline_tails_01": "#EF553B",
    "DyS_hellinger": "#fc03d3",
    "DyS_topsoe": "#fc03d3",
    "QuaDapt_DyS": "#fc03d3",
    "baseline_lite": "#19D3F3",
    "mfe": "#19D3F3",
    "qderiv_lite": "#19D3F3",
    "MiniRocket": "#19D3F3",
    "tsfresh": "#19D3F3",
    "catch22": "#19D3F3",
    "divergence_lite": "#19D3F3",
    "tail_lite": "#19D3F3",
    "shape_lite": "#19D3F3"
}

# ============================
# PLOT
# ============================

fig = px.box(
    df,
    x="modelo",
    y="erro",
    category_orders={"modelo": order},
    points="all",
    color="modelo",
    color_discrete_map=color_map
)

fig.update_traces(
    jitter=0.35,
    marker=dict(size=4, opacity=0.6),
)

fig.update_layout(
    title="Comparação de erro entre modelos (ordenado do melhor ao pior)",
    xaxis_title="Modelo",
    yaxis_title="Erro absoluto |prev_pred − prev_real|",
    template="simple_white",
    width=950,
    height=450,
    showlegend=True
)

fig